# Model Evaluation (Base vs. v1 vs. v2 vs. v3 RAG-Aware Comparison)

Welcome to the **Comprehensive Model Evaluation Notebook**! In this session, we will:
1. Load the **Base**, **Fine-Tuned v1**, **Fine-Tuned v2**, and **Fine-Tuned v3 (RAG-Aware)** versions of Qwen and Llama.
2. Run predictions on our unseen test datasets (`test.json` and `test_v3.json`) and calculate NLP evaluation metrics (BLEU, ROUGE-1, ROUGE-2, ROUGE-L).
3. Compare progress across all 4 stages of model evolution.
4. Print side-by-side responses (Base vs. v1 vs. v2 vs. v3) for detailed qualitative analysis.

---  
## Step 1: Mount Google Drive & Install Required Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl rouge-score nltk pandas matplotlib

---  
## Step 2: Initialize Workspace & Sync Files

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Project folder not found on Drive. Defaulting target back to: {gdrive_dir}")
else:
    print(f"[+] Detected active Drive folder: {gdrive_dir}")

for d in ["data/raw", "data/processed", "configs", "src", "models/evaluation"]:
    os.makedirs(os.path.join(project_dir, d), exist_ok=True)

drive_processed = os.path.join(gdrive_dir, "data", "processed")
local_processed = os.path.join(project_dir, "data", "processed")
if os.path.isdir(drive_processed):
    print("[*] Copying data splits from Google Drive...")
    !cp -v "{drive_processed}/"*.json "{local_processed}/" 2>/dev/null || true

drive_eval = os.path.join(gdrive_dir, "models", "evaluation")
local_eval = os.path.join(project_dir, "models", "evaluation")
if os.path.isdir(drive_eval):
    print("[*] Copying previous evaluation results from Google Drive...")
    !cp -v "{drive_eval}/"*.json "{local_eval}/" 2>/dev/null || true

print("[+] Workspace setup complete.")

---  
## Step 3: Write Latest Evaluation Script to Workspace

In [ ]:
evaluate_code = "import os\nimport argparse\nimport json\nimport torch\nimport random\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom rouge_score import rouge_scorer\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\n\n# Download NLTK data if not present (handled quietly)\ntry:\n    nltk.data.find('tokenizers/punkt')\nexcept LookupError:\n    nltk.download('punkt', quiet=True)\ntry:\n    nltk.data.find('tokenizers/punkt_tab')\nexcept LookupError:\n    nltk.download('punkt_tab', quiet=True)\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate fine-tuned model against reference dataset.\")\n    parser.add_argument(\n        \"--model_id\",\n        type=str,\n        required=True,\n        help=\"Hugging Face base model identifier (e.g. Qwen/Qwen2.5-7B-Instruct or meta-llama/Meta-Llama-3-8B-Instruct)\"\n    )\n    parser.add_argument(\n        \"--adapter_dir\",\n        type=str,\n        default=None,\n        help=\"Path to the LoRA adapter directory. If None, evaluates the base model only.\"\n    )\n    parser.add_argument(\n        \"--test_file\",\n        type=str,\n        default=\"data/processed/test.json\",\n        help=\"Path to the test JSON file.\"\n    )\n    parser.add_argument(\n        \"--output_file\",\n        type=str,\n        required=True,\n        help=\"Path to save the evaluation results JSON file.\"\n    )\n    parser.add_argument(\n        \"--num_samples\",\n        type=int,\n        default=100,\n        help=\"Number of random samples to evaluate (default: 100).\"\n    )\n    parser.add_argument(\n        \"--seed\",\n        type=int,\n        default=42,\n        help=\"Random seed for reproducibility.\"\n    )\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    \n    print(\"\\n==============================================\")\n    print(f\"[*] Base Model: {args.model_id}\")\n    print(f\"[*] Adapter Path: {args.adapter_dir}\")\n    print(f\"[*] Output Path: {args.output_file}\")\n    print(\"==============================================\\n\")\n    \n    token = os.environ.get(\"HF_TOKEN\") or True\n\n    # 1. Load Tokenizer\n    print(\"[*] Loading tokenizer...\")\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True, token=token)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    # 2. Load Model in 4-bit Quantization (to fit in T4 GPU VRAM)\n    print(\"[*] Loading base model in 4-bit quantization...\")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16\n    )\n    \n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16,\n        token=token\n    )\n    \n    # Force all bfloat16 parameters and buffers in the base model to float16 to prevent bfloat16 propagation\n    for name, param in base_model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in base_model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    # 3. Load LoRA Adapter if provided\n    if args.adapter_dir:\n        print(f\"[*] Loading LoRA adapter from {args.adapter_dir}...\")\n        model = PeftModel.from_pretrained(base_model, args.adapter_dir)\n    else:\n        print(\"[*] No adapter provided. Evaluating raw base model.\")\n        model = base_model\n        \n    model.eval()\n    \n    # 4. Load Test Dataset\n    print(f\"[*] Loading test file: {args.test_file}...\")\n    if not os.path.exists(args.test_file):\n        raise FileNotFoundError(f\"Test file not found: {args.test_file}\")\n        \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n        \n    if len(test_data) > args.num_samples:\n        print(f\"[*] Sampling {args.num_samples} records from {len(test_data)} total test records.\")\n        # Ensure repeatable sampling using seeded random\n        test_samples = random.sample(test_data, args.num_samples)\n    else:\n        print(f\"[*] Using all {len(test_data)} test records.\")\n        test_samples = test_data\n        \n    # 5. Setup Scorers\n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    # 6. Evaluation Generation Loop\n    print(\"\\n[*] Starting text generation and evaluation...\")\n    for idx, sample in enumerate(test_samples):\n        instruction = sample[\"instruction\"]\n        reference = sample[\"response\"]\n        context = sample.get(\"context\", \"\").strip()\n        \n        # Build prompt using SFT instruction-tuning prompt template (RAG-aware)\n        if context:\n            prompt = (\n                f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n                f\"Write a response that appropriately completes the request.\\n\\n\"\n                f\"### Instruction:\\n{instruction}\\n\\n\"\n                f\"### Context:\\n{context}\\n\\n\"\n                f\"### Response:\\n\"\n            )\n        else:\n            prompt = f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n### Instruction:\\n{instruction}\\n\\n### Response:\\n\"\n        \n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\")\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=150,\n                temperature=0.7,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n            \n        # Slice outputs to retrieve only the generated completion (ignoring prompt tokens)\n        prompt_len = inputs.input_ids.shape[1]\n        generation_tokens = outputs[0][prompt_len:]\n        prediction = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        \n        # Compute ROUGE\n        rouge_scores = rouge_scorer_inst.score(reference, prediction)\n        r1 = rouge_scores['rouge1'].fmeasure\n        r2 = rouge_scores['rouge2'].fmeasure\n        rl = rouge_scores['rougeL'].fmeasure\n        \n        # Compute BLEU (word level)\n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(prediction.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        # Accumulate scores\n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"instruction\": instruction,\n            \"reference\": reference,\n            \"prediction\": prediction,\n            \"metrics\": {\n                \"rouge1\": r1,\n                \"rouge2\": r2,\n                \"rougeL\": rl,\n                \"bleu\": bleu,\n                \"length\": len(prediction)\n            }\n        })\n        \n        if (idx + 1) % 10 == 0 or (idx + 1) == len(test_samples):\n            print(f\"    Processed {idx + 1}/{len(test_samples)} samples...\")\n            \n    # Calculate Summary Scores\n    num_evaluated = len(test_samples)\n    summary = {\n        \"mean_rouge1\": total_r1 / num_evaluated,\n        \"mean_rouge2\": total_r2 / num_evaluated,\n        \"mean_rougeL\": total_rl / num_evaluated,\n        \"mean_bleu\": total_bleu / num_evaluated\n    }\n    \n    output_data = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": args.adapter_dir,\n        \"summary\": summary,\n        \"results\": results\n    }\n    \n    # 7. Write Results\n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(output_data, f, ensure_ascii=False, indent=2)\n        \n    print(\"\\n========================= SUMMARY =========================\")\n    print(f\"[+] ROUGE-1 F-Measure: {summary['mean_rouge1']:.4f}\")\n    print(f\"[+] ROUGE-2 F-Measure: {summary['mean_rouge2']:.4f}\")\n    print(f\"[+] ROUGE-L F-Measure: {summary['mean_rougeL']:.4f}\")\n    print(f\"[+] BLEU Score:        {summary['mean_bleu']:.4f}\")\n    print(\"===========================================================\\n\")\n    print(f\"[+] Detailed evaluation records saved to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate.py", "w", encoding="utf-8") as f:
    f.write(evaluate_code)
print("[+] src/evaluate.py (with Hugging Face token support) successfully written.")

---  
## Step 4: Evaluate Qwen Models (Base vs. FT v1 vs. FT v2 vs. FT v3)

In [ ]:
# 1. Evaluate Base Qwen Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_base_results.json \
    --num_samples 100

In [ ]:
# 2. Evaluate Fine-Tuned Qwen v1 Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "{gdrive_dir}/models/qwen_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_finetuned_results.json \
    --num_samples 100

In [ ]:
# 3. Evaluate Fine-Tuned Qwen v2 Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "{gdrive_dir}/models/qwen_v2" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_v2_results.json \
    --num_samples 100

In [ ]:
# 4. Evaluate Fine-Tuned Qwen v3 Model (RAG-Aware)
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "{gdrive_dir}/models/qwen_v3" \
    --test_file /content/Retail/data/processed/test_v3.json \
    --output_file /content/Retail/models/evaluation/qwen_v3_results.json \
    --num_samples 100

---  
## Step 5: Evaluate Llama Models (Base vs. FT v1 vs. FT v2 vs. FT v3)

In [ ]:
from google.colab import userdata
import os

for secret_name in ['HF_TOKEN', 'HF_TOKEI', 'HF_token', 'hf_token']:
    try:
        val = userdata.get(secret_name)
        if val:
            os.environ['HF_TOKEN'] = val
            print(f"[+] Successfully loaded HF_TOKEN from Colab Secrets ({secret_name}).")
            break
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    print("[-] Warning: HF_TOKEN secret not found in Colab Secrets.")

# 1. Evaluate Base Llama Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_base_results.json \
    --num_samples 100

In [ ]:
# 2. Evaluate Fine-Tuned Llama v1 Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_finetuned_results.json \
    --num_samples 100

In [ ]:
# 3. Evaluate Fine-Tuned Llama v2 Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v2" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_v2_results.json \
    --num_samples 100

In [ ]:
# 4. Evaluate Fine-Tuned Llama v3 Model (RAG-Aware)
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v3" \
    --test_file /content/Retail/data/processed/test_v3.json \
    --output_file /content/Retail/models/evaluation/llama_v3_results.json \
    --num_samples 100

---  
## Step 6: Progressive Performance Plot (Base vs. v1 vs. v2 vs. v3) & Side-by-Side QA

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import os

eval_dir = "/content/Retail/models/evaluation"
models = {
    "Qwen Base": "qwen_base_results.json",
    "Qwen FT v1": "qwen_finetuned_results.json",
    "Qwen FT v2": "qwen_v2_results.json",
    "Qwen FT v3 (RAG)": "qwen_v3_results.json",
    "Llama Base": "llama_base_results.json",
    "Llama FT v1": "llama_finetuned_results.json",
    "Llama FT v2": "llama_v2_results.json",
    "Llama FT v3 (RAG)": "llama_v3_results.json"
}

metrics_summary = {}
for name, fname in models.items():
    fpath = os.path.join(eval_dir, fname)
    if os.path.exists(fpath):
        with open(fpath, "r") as f:
            data = json.load(f)
            metrics_summary[name] = data["summary"]

if metrics_summary:
    df = pd.DataFrame(metrics_summary).T
    print("\n===================== COMPLETE PROGRESSIVE SCORES =====================")
    print(df.round(4))
    print("========================================================================\n")
    
    fig, ax = plt.subplots(figsize=(14, 7))
    df[["mean_rouge1", "mean_rougeL", "mean_bleu"]].plot(kind="bar", ax=ax)
    ax.set_title("4-Stage Progressive Performance (Base vs. v1 vs. v2 vs. v3 RAG)")
    ax.set_ylabel("Score (Higher is Better)")
    ax.set_xticklabels(df.index, rotation=25, ha="right")
    ax.legend(["ROUGE-1", "ROUGE-L", "BLEU"])
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("[X] No evaluation metrics found.")

In [ ]:
# Show side-by-side responses for interactive validation (Base vs v1 vs v2 vs v3)
import pandas as pd
import os
import json

qwen_base_path = os.path.join(eval_dir, "qwen_base_results.json")
qwen_v1_path = os.path.join(eval_dir, "qwen_finetuned_results.json")
qwen_v2_path = os.path.join(eval_dir, "qwen_v2_results.json")
qwen_v3_path = os.path.join(eval_dir, "qwen_v3_results.json")

if os.path.exists(qwen_v3_path):
    with open(qwen_v3_path) as f: v3_data = json.load(f)["results"]
    base_data = json.load(open(qwen_base_path))["results"] if os.path.exists(qwen_base_path) else []
    v1_data = json.load(open(qwen_v1_path))["results"] if os.path.exists(qwen_v1_path) else []
    v2_data = json.load(open(qwen_v2_path))["results"] if os.path.exists(qwen_v2_path) else []
    
    compare_list = []
    for i in range(min(5, len(v3_data))):
        item = {
            "Instruction": v3_data[i].get("instruction", ""),
            "Context (RAG Passage)": v3_data[i].get("context", ""),
            "Reference": v3_data[i].get("reference", ""),
            "Base Model": base_data[i]["prediction"] if i < len(base_data) else "N/A",
            "FT v1 Model": v1_data[i]["prediction"] if i < len(v1_data) else "N/A",
            "FT v2 Model": v2_data[i]["prediction"] if i < len(v2_data) else "N/A",
            "FT v3 (RAG) Model": v3_data[i]["prediction"]
        }
        compare_list.append(item)
    
    df_compare = pd.DataFrame(compare_list)
    pd.set_option('display.max_colwidth', None)
    display(df_compare.style.set_properties(**{'text-align': 'left'}))
else:
    print("[!] Run Qwen v3 evaluation first to view 4-way comparison table.")

---  
## Step 7: Save Evaluation Results & Reports to Google Drive

In [ ]:
print(f"[*] Saving evaluation reports to Drive: {gdrive_dir}...")
local_eval_path = "/content/Retail/models/evaluation"
drive_eval_path = os.path.join(gdrive_dir, "models", "evaluation")
os.makedirs(drive_eval_path, exist_ok=True)
!cp -v "{local_eval_path}/"*.json "{drive_eval_path}/"
print("[+] Backup complete.")